In [0]:
# Import the required Delta Lake, Python and PySpark components

from datetime import datetime

from delta.tables import DeltaTable

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    current_timestamp,
    lit,
    max as spark_max,
    round as spark_round,
    sum as spark_sum
)

In [0]:
# Receive pipeline parameters and define table names

dbutils.widgets.text(
    "batch_id",
    "2009-12",
    "Batch ID"
)

dbutils.widgets.text(
    "run_id",
    "manual-run-001",
    "Run ID"
)

batch_id = dbutils.widgets.get("batch_id")
run_id = dbutils.widgets.get("run_id")

control_table = "online_retail_aws.control.pipeline_runs"
silver_table = "online_retail_aws.silver.transactions_clean"

gold_table = "online_retail_aws.gold.product_sales_summary"
gold_table_path = "s3://online-retail-databricks/tables/gold/product_sales_summary/"

layer_name = "gold"

print(f"Run ID: {run_id}")
print(f"Batch ID: {batch_id}")
print(f"Silver table: {silver_table}")
print(f"Gold table: {gold_table}")

Run ID: demo-2010-02
Batch ID: 2010-02
Silver table: online_retail_aws.silver.transactions_clean
Gold table: online_retail_aws.gold.product_sales_summary


In [0]:
# Validate that batch_id is a valid month in exact YYYY-MM format

try:
    batch_month = datetime.strptime(batch_id, "%Y-%m")

    if batch_month.strftime("%Y-%m") != batch_id:
        raise ValueError

except ValueError as error:
    raise ValueError(
        f"Invalid batch_id: {batch_id}. Expected YYYY-MM."
    ) from error

else:
    print(f"Valid batch ID: {batch_id}")

Valid batch ID: 2010-02


In [0]:
# Record that Gold processing has started

if not spark.catalog.tableExists(control_table):
    raise ValueError(
        f"Control table does not exist: {control_table}"
    )


started_audit_df = (
    spark.range(1)
    .select(
        lit(run_id).alias("run_id"),
        lit(batch_id).alias("batch_id"),
        lit(layer_name).alias("layer_name"),
        lit("STARTED").alias("status"),
        current_timestamp().alias("start_timestamp"),
        lit(None).cast("timestamp").alias("end_timestamp"),
        lit(None).cast("long").alias("input_row_count"),
        lit(None).cast("long").alias("output_row_count"),
        lit(None).cast("string").alias("error_message")
    )
)


control_delta_table = DeltaTable.forName(
    spark,
    control_table
)


(
    control_delta_table.alias("target")
    .merge(
        started_audit_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.batch_id = source.batch_id
        AND target.layer_name = source.layer_name
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(
    f"Gold processing started for batch {batch_id}, "
    f"run {run_id}."
)

Gold processing started for batch 2010-02, run demo-2010-02.


In [0]:
# Read the selected Silver batch and retain positive sales

if not spark.catalog.tableExists(silver_table):
    raise ValueError(
        f"Silver table does not exist: {silver_table}"
    )


silver_batch_df = (
    spark.table(silver_table)
    .filter(col("batch_id") == batch_id)
)

silver_batch_row_count = silver_batch_df.count()


if silver_batch_row_count == 0:
    raise ValueError(
        f"Silver contains no records for batch {batch_id}."
    )


positive_sales_batch_df = (
    silver_batch_df
    .filter(col("is_positive_sale"))
)

positive_sales_row_count = (
    positive_sales_batch_df.count()
)


if positive_sales_row_count == 0:
    raise ValueError(
        f"Batch {batch_id} contains no positive-sale records."
    )


print(f"Silver batch rows: {silver_batch_row_count:,}")
print(
    f"Positive-sale records: "
    f"{positive_sales_row_count:,}"
)

Silver batch rows: 29,058
Positive-sale records: 27,952


In [0]:
# Aggregate positive sales at batch and product grain

gold_batch_df = (
    positive_sales_batch_df
    .groupBy(
        "batch_id",
        "stock_code"
    )
    .agg(
        spark_max("description").alias("description"),

        spark_sum("quantity").alias(
            "total_quantity_sold"
        ),

        spark_round(
            spark_sum("line_total"),
            2
        ).alias("total_revenue"),

        countDistinct("invoice").alias(
            "distinct_invoice_count"
        ),

        countDistinct("customer_id").alias(
            "distinct_customer_count"
        )
    )
)


gold_batch_row_count = gold_batch_df.count()

print(f"Gold batch rows: {gold_batch_row_count:,}")

Gold batch rows: 2,577


In [0]:
# Reconcile positive-sale totals between Silver and prepared Gold

silver_totals = (
    positive_sales_batch_df
    .agg(
        spark_sum("quantity").alias(
            "silver_total_quantity"
        ),

        spark_round(
            spark_sum("line_total"),
            2
        ).alias("silver_total_revenue")
    )
    .first()
)


gold_totals = (
    gold_batch_df
    .agg(
        spark_sum("total_quantity_sold").alias(
            "gold_total_quantity"
        ),

        spark_round(
            spark_sum("total_revenue"),
            2
        ).alias("gold_total_revenue")
    )
    .first()
)


silver_total_quantity = (
    silver_totals["silver_total_quantity"]
)

silver_total_revenue = (
    silver_totals["silver_total_revenue"]
)

gold_total_quantity = (
    gold_totals["gold_total_quantity"]
)

gold_total_revenue = (
    gold_totals["gold_total_revenue"]
)


quantities_match = (
    silver_total_quantity == gold_total_quantity
)

revenues_match = (
    abs(silver_total_revenue - gold_total_revenue)
    <= 0.01
)


print(
    f"Silver total quantity: "
    f"{silver_total_quantity:,}"
)

print(
    f"Gold total quantity: "
    f"{gold_total_quantity:,}"
)

print(f"Quantities match: {quantities_match}")

print(
    f"Silver total revenue: "
    f"{silver_total_revenue:,.2f}"
)

print(
    f"Gold total revenue: "
    f"{gold_total_revenue:,.2f}"
)

print(f"Revenues match: {revenues_match}")


if not quantities_match:
    raise ValueError(
        f"Silver and Gold quantities do not match "
        f"for batch {batch_id}."
    )

if not revenues_match:
    raise ValueError(
        f"Silver and Gold revenues do not match "
        f"for batch {batch_id}."
    )

print(
    f"Prepared Gold batch {batch_id} passed reconciliation."
)

Silver total quantity: 381,879
Gold total quantity: 381,879
Quantities match: True
Silver total revenue: 551,504.72
Gold total revenue: 551,504.72
Revenues match: True
Prepared Gold batch 2010-02 passed reconciliation.


In [0]:
# Validate the batch_id and stock_code MERGE key

duplicate_merge_key_group_count = (
    gold_batch_df
    .groupBy(
        "batch_id",
        "stock_code"
    )
    .agg(
        count("*").alias("record_count")
    )
    .filter(col("record_count") > 1)
    .count()
)


null_merge_key_count = (
    gold_batch_df
    .filter(
        col("batch_id").isNull()
        | col("stock_code").isNull()
    )
    .count()
)


print(
    "Duplicate MERGE-key groups: "
    f"{duplicate_merge_key_group_count:,}"
)

print(
    f"Null MERGE-key records: "
    f"{null_merge_key_count:,}"
)


if duplicate_merge_key_group_count > 0:
    raise ValueError(
        f"Duplicate Gold MERGE keys exist in "
        f"batch {batch_id}."
    )

if null_merge_key_count > 0:
    raise ValueError(
        f"Null Gold MERGE keys exist in "
        f"batch {batch_id}."
    )

print(
    f"Gold MERGE key passed validation "
    f"for batch {batch_id}."
)

Duplicate MERGE-key groups: 0
Null MERGE-key records: 0
Gold MERGE key passed validation for batch 2010-02.


In [0]:
# Synchronize the current product-summary batch with the Gold table

if spark.catalog.tableExists(gold_table):
    gold_delta_table = DeltaTable.forName(
        spark,
        gold_table
    )

    (
        gold_delta_table.alias("target")
        .merge(
            gold_batch_df.alias("source"),
            """
            target.batch_id = source.batch_id
            AND target.stock_code = source.stock_code
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .whenNotMatchedBySourceDelete(
            condition=f"target.batch_id = '{batch_id}'"
        )
        .execute()
    )

    print(
        f"Synchronized batch {batch_id} "
        f"in {gold_table}."
    )

else:
    (
        gold_batch_df
        .write
        .format("delta")
        .option("path", gold_table_path)
        .saveAsTable(gold_table)
    )

    print(
        f"Created {gold_table} at {gold_table_path} "
        f"with batch {batch_id}."
    )

Synchronized batch 2010-02 in online_retail_aws.gold.product_sales_summary.


In [0]:
# Verify the stored Gold batch and its MERGE keys

stored_gold_batch_df = (
    spark.table(gold_table)
    .filter(col("batch_id") == batch_id)
)

stored_gold_batch_row_count = (
    stored_gold_batch_df.count()
)


stored_duplicate_key_group_count = (
    stored_gold_batch_df
    .groupBy(
        "batch_id",
        "stock_code"
    )
    .agg(
        count("*").alias("record_count")
    )
    .filter(col("record_count") > 1)
    .count()
)


prepared_keys_missing_from_gold = (
    gold_batch_df
    .select(
        "batch_id",
        "stock_code"
    )
    .join(
        stored_gold_batch_df.select(
            "batch_id",
            "stock_code"
        ),
        on=[
            "batch_id",
            "stock_code"
        ],
        how="left_anti"
    )
    .count()
)


obsolete_stored_gold_keys = (
    stored_gold_batch_df
    .select(
        "batch_id",
        "stock_code"
    )
    .join(
        gold_batch_df.select(
            "batch_id",
            "stock_code"
        ),
        on=[
            "batch_id",
            "stock_code"
        ],
        how="left_anti"
    )
    .count()
)


print(
    f"Stored Gold batch rows: "
    f"{stored_gold_batch_row_count:,}"
)

print(
    "Stored duplicate MERGE-key groups: "
    f"{stored_duplicate_key_group_count:,}"
)

print(
    "Prepared keys missing from Gold: "
    f"{prepared_keys_missing_from_gold:,}"
)

print(
    "Obsolete stored Gold keys: "
    f"{obsolete_stored_gold_keys:,}"
)


if stored_gold_batch_row_count != gold_batch_row_count:
    raise ValueError(
        f"Stored Gold row count for batch {batch_id} "
        f"does not match the prepared Gold row count."
    )

if stored_duplicate_key_group_count > 0:
    raise ValueError(
        f"Duplicate keys exist in stored Gold "
        f"batch {batch_id}."
    )

if prepared_keys_missing_from_gold > 0:
    raise ValueError(
        f"{prepared_keys_missing_from_gold} prepared Gold "
        f"keys are missing from the stored table."
    )

if obsolete_stored_gold_keys > 0:
    raise ValueError(
        f"{obsolete_stored_gold_keys} obsolete keys remain "
        f"in stored Gold batch {batch_id}."
    )

print(
    f"Stored Gold batch {batch_id} passed key validation."
)

Stored Gold batch rows: 2,577
Stored duplicate MERGE-key groups: 0
Prepared keys missing from Gold: 0
Obsolete stored Gold keys: 0
Stored Gold batch 2010-02 passed key validation.


In [0]:
# Reconcile positive Silver sales with the stored Gold batch

stored_gold_totals = (
    stored_gold_batch_df
    .agg(
        spark_sum("total_quantity_sold").alias(
            "stored_total_quantity"
        ),

        spark_round(
            spark_sum("total_revenue"),
            2
        ).alias("stored_total_revenue")
    )
    .first()
)


stored_total_quantity = (
    stored_gold_totals["stored_total_quantity"]
)

stored_total_revenue = (
    stored_gold_totals["stored_total_revenue"]
)


stored_quantities_match = (
    silver_total_quantity == stored_total_quantity
)

stored_revenues_match = (
    abs(silver_total_revenue - stored_total_revenue)
    <= 0.01
)


print(
    f"Silver total quantity: "
    f"{silver_total_quantity:,}"
)

print(
    f"Stored Gold total quantity: "
    f"{stored_total_quantity:,}"
)

print(
    f"Stored quantities match: "
    f"{stored_quantities_match}"
)

print(
    f"Silver total revenue: "
    f"{silver_total_revenue:,.2f}"
)

print(
    f"Stored Gold total revenue: "
    f"{stored_total_revenue:,.2f}"
)

print(
    f"Stored revenues match: "
    f"{stored_revenues_match}"
)


if not stored_quantities_match:
    raise ValueError(
        f"Stored Gold quantity does not reconcile with "
        f"Silver for batch {batch_id}."
    )

if not stored_revenues_match:
    raise ValueError(
        f"Stored Gold revenue does not reconcile with "
        f"Silver for batch {batch_id}."
    )

print(
    f"Stored Gold batch {batch_id} "
    f"passed reconciliation."
)

Silver total quantity: 381,879
Stored Gold total quantity: 381,879
Stored quantities match: True
Silver total revenue: 551,504.72
Stored Gold total revenue: 551,504.72
Stored revenues match: True
Stored Gold batch 2010-02 passed reconciliation.


In [0]:
# Mark Gold processing as successful in the control table

control_delta_table.update(
    condition=(
        (col("run_id") == run_id)
        & (col("batch_id") == batch_id)
        & (col("layer_name") == layer_name)
    ),
    set={
        "status": lit("SUCCESS"),
        "end_timestamp": current_timestamp(),
        "input_row_count": lit(silver_batch_row_count),
        "output_row_count": lit(
            stored_gold_batch_row_count
        ),
        "error_message": lit(None).cast("string")
    }
)

print(
    f"Audit completed successfully for Gold, "
    f"batch {batch_id}, run {run_id}."
)

Audit completed successfully for Gold, batch 2010-02, run demo-2010-02.
